# KNN Variants for Class Imbalance
## ML1 Practical Assignment 2025/2026

> For full algorithm derivations see [`algorithm_design.ipynb`](algorithm_design.ipynb).  
> For raw benchmark runs see [`phase1_baseline.ipynb`](phase1_baseline.ipynb) and [`phase2_benchmark.ipynb`](phase2_benchmark.ipynb).

---
## 1. Selected Algorithm and Data Characteristic

**Algorithm:** K-Nearest Neighbours (KNN) — implemented from scratch in `src/algorithms/knn_base.py`.

**Data characteristic:** class imbalance in binary classification.

**Why KNN is affected:** KNN classifies by majority vote over the k nearest neighbours. Under class imbalance the majority class dominates neighbourhoods mechanically — even if a query point lies near the minority class boundary, the majority class wins the vote. The bias grows with the imbalance ratio (IR) and with k.

*(Load and display the IR distribution figure here)*

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from src.utils.config import load_config, get_project_root

cfg = load_config()
FIG_DIR = get_project_root() / cfg['paths']['results_figures']
TAB_DIR = get_project_root() / cfg['paths']['results_tables']

%matplotlib inline
print('Setup complete.')

### 1.1 Baseline Performance

Baseline KNN (k=5) and CV-selected `KNNOptK` evaluated on 40 benchmark datasets (10-fold stratified CV).

In [ ]:
# Load Phase 1 results and display summary table
baseline = pd.read_csv(TAB_DIR / 'phase1_optk.csv')
baseline.head()

In [ ]:
# IR vs G-mean scatter
fig, ax = plt.subplots(figsize=(6, 4))
# TODO: scatter IR vs G-mean for k=5 and OptK
plt.tight_layout()
plt.show()

---
## 2. Proposal — KNNFairRank

**Core idea:** standard KNN compares minority rank-1 against majority rank-1 — an unfair comparison when the minority class is underrepresented. KNNFairRank corrects this by comparing minority rank-1 against majority rank-$k_\text{eff}$, where $k_\text{eff}$ is derived from the class ratio.

**Derivation summary:** under a homogeneous Poisson process, the fair majority rank that equalises expected distances is $k_\text{eff} = N_\text{maj}/N_\text{min} = r$. A multi-rank vote across ranks 1 to $r$ reduces variance.

> Full derivation in [`algorithm_design.ipynb §2`](algorithm_design.ipynb).

**Variants:**
| Algorithm | Change over v3 | Targets |
|---|---|---|
| `KNNFairRank` (v3) | theoretical baseline | — |
| `KNNFairRankMagnitude` | continuous distance-weighted votes | F3 |
| `KNNFairRankCV` | CV-tuned exponent α on $k_\text{eff}=r^\alpha$ | F4 |

---
## 3. Empirical Study

### 3.1 Experimental Setup

- **Datasets:** 40 non-degenerate datasets from the `class_imbalance` suite (see Phase 1 notebook for exclusion criteria)
- **Evaluation:** 10-fold stratified cross-validation, repeated N times
- **Metrics:** F1 (minority class), G-mean, ROC-AUC
- **Baselines:** `KNNOptK` (CV k), `SMOTE+KNN` (industry standard)
- **Statistical tests:** Friedman + pairwise Wilcoxon with Holm correction

### 3.2 Overall Results

In [ ]:
# Load and display summary table
bench = pd.read_csv(TAB_DIR / 'benchmark_summary.csv')
bench

In [ ]:
# Boxplot figure
img = mpimg.imread(FIG_DIR / 'benchmark_boxplots.png')
plt.figure(figsize=(10, 4)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

### 3.3 Performance by Imbalance Severity

In [ ]:
img = mpimg.imread(FIG_DIR / 'benchmark_by_ir_quartile.png')
plt.figure(figsize=(10, 4)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

### 3.4 Statistical Tests

In [ ]:
wilcox = pd.read_csv(TAB_DIR / 'wilcoxon_vs_KNNFairRank_geometric_mean.csv')
wilcox

In [ ]:
img = mpimg.imread(FIG_DIR / 'average_ranks.png')
plt.figure(figsize=(8, 4)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

### 3.5 Bootstrap Confidence Intervals

In [ ]:
img = mpimg.imread(FIG_DIR / 'bootstrap_ci_forest.png')
plt.figure(figsize=(8, 5)); plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

---
## 4. Analysis of Results

*(Write discussion here — what the results show, where FairRank wins/loses, IR dependence, statistical significance)*

---
## 5. Conclusions and Future Work

**Conclusions:**
- KNNFairRankCV achieves the best average rank across metrics, confirming that a data-driven α exponent closes the reliability gap of the theoretical v3.
- KNNFairRank (v3) achieves the highest mean G-mean but with higher dataset-to-dataset variance.
- Both variants outperform SMOTE+KNN on high-IR datasets.

**Future work:**
- Modification E (direct density-ratio): generalises v3 by replacing the rank proxy with explicit k-NN density estimates — shown in `algorithm_design.ipynb §8` to subsume v3 as a special case.
- Topology-aware correction: use persistent homology to estimate local density structure, with confidence weighting to handle minority topology noise — see `algorithm_design.ipynb §9`.